# Change Detection with NDVI Differencing

**Purpose:** Compute the vegetation change between two dates using NDVI from Sentinel-2 via Google Earth Engine.

**What this notebook teaches:**
- How NDVI is computed from Sentinel-2 bands in GEE
- How to compare two dates and compute a difference image
- How to threshold and colorize the change for visual interpretation
- How to compute summary statistics (mean change, area gained, area lost)

**Data sources:** Google Earth Engine — Sentinel-2 Surface Reflectance (COPERNICUS/S2_SR_HARMONIZED)

**Libraries required:** earthengine-api, geemap, matplotlib, numpy

**Expected runtime:** 3-5 minutes (GEE queries run server-side; only the final image downloads)

**Key outputs:**
- NDVI map for Date 1
- NDVI map for Date 2
- Change map (NDVI Date 2 minus NDVI Date 1)
- Summary statistics: mean change, area gained, area lost

## What is NDVI?

NDVI stands for Normalized Difference Vegetation Index. It is the most widely used index for measuring vegetation health from satellite imagery.

The formula: **(NIR - Red) / (NIR + Red)**

- Healthy vegetation absorbs red light strongly (for photosynthesis) and reflects near-infrared strongly (a structural property of plant cells).
- Bare soil reflects both red and NIR at roughly similar levels.
- Water absorbs both red and NIR almost completely.

NDVI ranges from -1 to +1. In practice:
- Dense healthy vegetation: 0.5 to 0.9
- Sparse or stressed vegetation: 0.2 to 0.5
- Bare soil: 0.1 to 0.25
- Water: near 0 or negative

**In Sentinel-2:** NIR is Band 8 (842 nm). Red is Band 4 (665 nm).

## What is Change Detection?

Change detection computes the pixel-by-pixel difference between two measurements of the same area at different times. When applied to NDVI:

- A positive difference means vegetation increased between Date 1 and Date 2 (greening, regrowth, crop growth)
- A negative difference means vegetation decreased (drought, deforestation, fire, harvest)
- Near-zero means the surface did not change significantly

This is one of the most operationally useful techniques in Earth observation. It is used for deforestation monitoring, post-fire damage assessment, crop yield estimation, and drought early warning.

In [ ]:
# Cell 1: Install and import libraries
# These libraries are already installed in the EOIL environment.
# Run this cell first to confirm everything loads without errors.

import ee
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

print('All libraries imported successfully.')

In [ ]:
# Cell 2: Authenticate and initialize Google Earth Engine
#
# This notebook uses interactive authentication (browser-based).
# The portal app uses a service account instead — this is the notebook-friendly equivalent.
#
# First time on a new machine: ee.Authenticate() opens a browser and asks you to log in.
# After that: ee.Initialize() uses the saved credentials automatically.

try:
    ee.Initialize(project='gen-lang-client-0093165324')
    print('GEE initialized successfully — using cached credentials.')
except Exception:
    print('No cached credentials — running authentication...')
    ee.Authenticate()
    ee.Initialize(project='gen-lang-client-0093165324')
    print('GEE initialized after authentication.')

In [ ]:
# Cell 3: Define the study area and two dates
#
# We are looking at the Sahel region of West Africa — specifically a section of
# southern Senegal and The Gambia. This region has strong NDVI seasonality:
# near-zero in the dry season (December-March) and high in the wet season (August-October).
# That contrast makes it a reliable test case — we should always see clear change.
#
# bbox format: [west, south, east, north] in decimal degrees

STUDY_REGION = {
    'name': 'Sahel — Southern Senegal / Gambia',
    'bbox': [-16.5, 12.5, -14.5, 14.5],   # [west, south, east, north]
    'date1': '2023-02-01',                  # dry season — expected low NDVI
    'date2': '2023-09-01',                  # wet season — expected high NDVI
}

bbox   = STUDY_REGION['bbox']
date1  = STUDY_REGION['date1']
date2  = STUDY_REGION['date2']
name   = STUDY_REGION['name']

# Convert bbox to a GEE geometry object
# GEE uses this to filter the image collection to only scenes that overlap the region
geometry = ee.Geometry.Rectangle(bbox)

print(f'Study region: {name}')
print(f'Bounding box: {bbox}')
print(f'Date 1: {date1}  (dry season)')
print(f'Date 2: {date2}  (wet season)')

In [ ]:
# Cell 4: Define the NDVI function and fetch images for both dates
#
# This is the core GEE pattern for this notebook.
# We filter the Sentinel-2 Surface Reflectance collection to:
#   - Only images that overlap the region
#   - Only images within a 30-day window of the target date
#   - Only images with less than 20% cloud cover
# Then we take the median of all qualifying images.
# Using the median (rather than a single image) removes most cloud contamination:
# clouds are bright and appear in different positions across scenes, so the median
# value for each pixel tends to be cloud-free.
#
# Sentinel-2 Surface Reflectance collection ID: COPERNICUS/S2_SR_HARMONIZED
# Bands: B4 = Red (665 nm), B8 = Near-Infrared (842 nm)
# NDVI = (B8 - B4) / (B8 + B4)

WINDOW_DAYS = 30   # search this many days either side of the target date
MAX_CLOUD   = 20   # ignore scenes with more than this % cloud cover

def compute_ndvi(image):
    """Add an NDVI band to a Sentinel-2 image.
    
    GEE's normalizedDifference() method computes (A - B) / (A + B).
    We rename the output band 'NDVI' so it is easy to select later.
    """
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)


def fetch_ndvi_composite(date_str, window_days=WINDOW_DAYS, max_cloud=MAX_CLOUD):
    """Fetch a cloud-free median NDVI composite for the study region near a given date.
    
    Filters Sentinel-2 SR collection by region, date window, and cloud cover.
    Computes NDVI for each image. Returns the median composite as a single GEE image.
    """
    date  = ee.Date(date_str)
    start = date.advance(-window_days, 'day')
    end   = date.advance(window_days,  'day')
    
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(geometry)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
        .map(compute_ndvi)
        .select('NDVI')
        .median()          # median across all qualifying scenes = cloud-free composite
    )
    return collection


print('Functions defined. Fetching NDVI composites from GEE...')
print('(No data is downloaded yet — GEE is building server-side computation graphs)')

ndvi_img1 = fetch_ndvi_composite(date1)
ndvi_img2 = fetch_ndvi_composite(date2)

print(f'\nNDVI composite 1 ready: centred on {date1}')
print(f'NDVI composite 2 ready: centred on {date2}')
print('Data will be downloaded in the next cells when we call getDownloadURL or sampleRectangle.')

In [ ]:
# Cell 5: Compute the NDVI difference image
#
# This is the change detection operation. One line of GEE code.
# ndvi_diff = Date2 NDVI minus Date1 NDVI
#
# Positive values: vegetation increased between date1 and date2
# Negative values: vegetation decreased
# Near-zero: no meaningful change

ndvi_diff = ndvi_img2.subtract(ndvi_img1).rename('NDVI_diff')

print('Difference image computed: ndvi_img2 minus ndvi_img1')
print()
print('What the values mean:')
print('  Positive (green on map):  vegetation increased from Date 1 to Date 2')
print('  Negative (red on map):    vegetation decreased from Date 1 to Date 2')
print('  Near zero (white on map): no significant change')
print()
print('Thresholds we will use for statistics:')
print('  Significant gain:  NDVI diff > +0.1')
print('  Significant loss:  NDVI diff < -0.1')
print('  No change:         -0.1 to +0.1')

In [ ]:
# Cell 6: Download pixel values for visualization
#
# GEE does not give you a numpy array directly. You have to call sampleRectangle()
# to download a small grid of pixel values. We use scale=500 (500m per pixel) to
# keep the download fast — this gives us a ~240x240 pixel grid for our 2-degree bbox.
#
# In the portal app, we do not download pixels at all. We use GEE tile URLs instead,
# which lets the browser request only the tiles it needs at the current zoom level.
# Here in the notebook we download a coarse grid for matplotlib visualization.

SCALE = 1000   # metres per pixel for the download — coarser = faster

def download_as_array(image, band_name, scale=SCALE):
    """Download a GEE image band to a numpy array via sampleRectangle.
    
    sampleRectangle() downloads all pixels within the geometry at the given scale.
    Returns a 2D numpy array of float values.
    """
    sample = image.select(band_name).sampleRectangle(
        region=geometry,
        defaultValue=0
    )
    values = sample.get(band_name).getInfo()
    return np.array(values, dtype=float)


print(f'Downloading NDVI Date 1 ({date1}) at {SCALE}m resolution...')
arr_ndvi1 = download_as_array(ndvi_img1, 'NDVI')
print(f'  Downloaded: {arr_ndvi1.shape} pixels, range {arr_ndvi1.min():.3f} to {arr_ndvi1.max():.3f}')

print(f'Downloading NDVI Date 2 ({date2}) at {SCALE}m resolution...')
arr_ndvi2 = download_as_array(ndvi_img2, 'NDVI')
print(f'  Downloaded: {arr_ndvi2.shape} pixels, range {arr_ndvi2.min():.3f} to {arr_ndvi2.max():.3f}')

print(f'Downloading NDVI difference at {SCALE}m resolution...')
arr_diff  = download_as_array(ndvi_diff, 'NDVI_diff')
print(f'  Downloaded: {arr_diff.shape} pixels, range {arr_diff.min():.3f} to {arr_diff.max():.3f}')

print('\nAll arrays downloaded.')

In [ ]:
# Cell 7: Visualize the three images side by side
#
# We display:
#   Left:   NDVI Date 1 — green colormap, range 0 to 0.9
#   Centre: NDVI Date 2 — same colormap for direct comparison
#   Right:  Change map  — diverging red-green colormap, centred on zero
#
# The diverging colormap is key to reading the change map:
# - Strong green: large NDVI gain (vegetation grew)
# - White: near-zero change
# - Strong red: large NDVI loss (vegetation died, burned, or was harvested)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'NDVI Change Detection — {name}', fontsize=14, fontweight='bold')

# NDVI Date 1
im1 = axes[0].imshow(arr_ndvi1, cmap='YlGn', vmin=0, vmax=0.9,
                      origin='upper', aspect='auto')
axes[0].set_title(f'NDVI — Date 1\n{date1}', fontsize=11)
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label='NDVI')

# NDVI Date 2
im2 = axes[1].imshow(arr_ndvi2, cmap='YlGn', vmin=0, vmax=0.9,
                      origin='upper', aspect='auto')
axes[1].set_title(f'NDVI — Date 2\n{date2}', fontsize=11)
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label='NDVI')

# Change map — symmetric range around zero for the diverging colormap
abs_max = max(abs(arr_diff.min()), abs(arr_diff.max()), 0.3)
im3 = axes[2].imshow(arr_diff, cmap='RdYlGn', vmin=-abs_max, vmax=abs_max,
                      origin='upper', aspect='auto')
axes[2].set_title(f'NDVI Change\n(Date 2 minus Date 1)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label='ΔNDVI')

plt.tight_layout()
plt.savefig('change_detection_sahel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure saved as change_detection_sahel.png')

In [ ]:
# Cell 8: Compute summary statistics
#
# We compute three statistics that summarize the change map:
#
# 1. Mean NDVI change — the average change across all pixels
#    Positive = net greening, negative = net browning
#
# 2. Area of significant gain — pixels where NDVI increased by more than +0.1
#    A threshold of 0.1 NDVI units is a common threshold for "real" vegetation change
#    (changes smaller than 0.1 can be noise or cloud contamination)
#
# 3. Area of significant loss — pixels where NDVI decreased by more than -0.1
#
# For the area calculation:
# Each pixel represents SCALE x SCALE metres = SCALE^2 square metres
# Convert to km2 by dividing by 1,000,000

CHANGE_THRESHOLD = 0.1

total_pixels  = arr_diff.size
gain_pixels   = np.sum(arr_diff > CHANGE_THRESHOLD)
loss_pixels   = np.sum(arr_diff < -CHANGE_THRESHOLD)
stable_pixels = total_pixels - gain_pixels - loss_pixels

# Area per pixel in km2
pixel_area_km2 = (SCALE / 1000) ** 2

area_gain   = gain_pixels   * pixel_area_km2
area_loss   = loss_pixels   * pixel_area_km2
area_stable = stable_pixels * pixel_area_km2
area_total  = total_pixels  * pixel_area_km2

mean_change = float(arr_diff.mean())

print('===== Change Detection Summary =====')
print(f'Study region:  {name}')
print(f'Date 1:        {date1}')
print(f'Date 2:        {date2}')
print()
print(f'Mean NDVI change:          {mean_change:+.4f}')
print(f'  Interpretation:          {"Net greening" if mean_change > 0 else "Net browning"}')
print()
print(f'Pixel resolution:          {SCALE} m')
print(f'Total area analysed:       {area_total:,.0f} km2')
print()
print(f'Significant gain (>{CHANGE_THRESHOLD}):  {area_gain:,.0f} km2  ({100*gain_pixels/total_pixels:.1f}%)')
print(f'Stable (-{CHANGE_THRESHOLD} to +{CHANGE_THRESHOLD}):        {area_stable:,.0f} km2  ({100*stable_pixels/total_pixels:.1f}%)')
print(f'Significant loss (<-{CHANGE_THRESHOLD}): {area_loss:,.0f} km2  ({100*loss_pixels/total_pixels:.1f}%)')

In [ ]:
# Cell 9: Distribution of change values
#
# A histogram of NDVI difference values shows you the full distribution of change.
# For the Sahel dry-to-wet comparison, we expect a strongly right-skewed distribution
# — most pixels show large positive gain because this is a seasonal extreme comparison.
# For a subtle change (e.g. two months in the same season), the distribution would be
# tightly clustered around zero with only small tails.

fig, ax = plt.subplots(figsize=(10, 5))

counts, bins, patches = ax.hist(
    arr_diff.flatten(), bins=60, edgecolor='none', alpha=0.85
)

# Colour the bars: red for loss, green for gain, grey for near-zero
for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge < -CHANGE_THRESHOLD:
        patch.set_facecolor('#d73027')
    elif left_edge > CHANGE_THRESHOLD:
        patch.set_facecolor('#1a9850')
    else:
        patch.set_facecolor('#b0b0b0')

ax.axvline(0, color='black', linewidth=1.2, linestyle='--', label='No change')
ax.axvline(-CHANGE_THRESHOLD, color='#d73027', linewidth=1, linestyle=':', label=f'Loss threshold ({-CHANGE_THRESHOLD})')
ax.axvline( CHANGE_THRESHOLD, color='#1a9850', linewidth=1, linestyle=':', label=f'Gain threshold (+{CHANGE_THRESHOLD})')

ax.set_xlabel('NDVI Difference (Date 2 minus Date 1)', fontsize=11)
ax.set_ylabel('Number of pixels', fontsize=11)
ax.set_title(f'Distribution of NDVI Change — {name}\n{date1} to {date2}', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('change_distribution_sahel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure saved as change_distribution_sahel.png')

## What the GEE tile URL approach looks like (used in the portal app)

In the notebook, we used `sampleRectangle()` to download a numpy array and visualize it with matplotlib. That works well for exploration but it downloads every pixel.

The portal app uses a different approach: **GEE tile URLs**.

When you call `image.getMapId({'min': -0.5, 'max': 0.5, 'palette': ['red','white','green']})`, GEE returns a URL like:

```
https://earthengine.googleapis.com/v1alpha/projects/.../{z}/{x}/{y}
```

This URL is a standard web map tile endpoint. When the browser pans or zooms, it requests only the tiles it needs. GEE renders each tile on demand at whatever zoom level the user is viewing. The app never downloads the full raster — only the ~256x256 pixel tiles visible on screen.

This is what makes GEE powerful for interactive apps: you can render a country-scale change map at 10m resolution and the app stays fast because only the visible tiles are ever computed.

In [ ]:
# Cell 10: Get a GEE tile URL for the change map
#
# This is what the portal app does under the hood.
# getMapId() returns a URL that Folium passes as a TileLayer.
# The browser fetches individual map tiles from that URL as the user pans and zooms.

vis_params = {
    'min':     -0.5,
    'max':      0.5,
    'palette': ['#d73027', '#f7f7f7', '#1a9850'],  # red, white, green
}

map_id = ndvi_diff.getMapId(vis_params)
tile_url = map_id['tile_fetcher'].url_format

print('GEE change map tile URL:')
print(tile_url[:100], '...')
print()
print('In the portal app, this URL is passed directly to Folium as a TileLayer.')
print('Folium renders the tiles in the browser. No pixel data moves through the app.')
print()

# Optional: show the URL in a clickable link (paste the z/x/y template into a WMTS viewer
# or use the Folium integration below)
print('Template format: {z}/{x}/{y} — Folium fills in the correct tile coordinates.')

## Key findings from this session

1. **NDVI differencing is simple.** One line of GEE code (`img2.subtract(img1)`) produces the change image. The complexity is in the data preparation — cloud filtering, median compositing, and choosing the right date window.

2. **The Sahel seasonal contrast is dramatic.** Comparing a dry season date (February) to a wet season date (September) typically shows NDVI gains of 0.4 to 0.6 across large areas. This is entirely driven by rainfall seasonality, not human-caused change.

3. **Threshold matters.** A change threshold of ±0.1 NDVI separates real vegetation change from noise. Smaller thresholds produce false positives from sensor calibration differences. Larger thresholds miss subtle deforestation or degradation events.

4. **GEE tile URLs are the right approach for interactive apps.** Downloading full rasters is slow and wasteful. The tile URL approach lets the browser fetch only what it displays.

## Questions for further investigation

1. **Intra-season change:** What does the change map look like when comparing two dates in the same season, 6 months apart? Can we detect a drought signal within a wet season?

2. **Deforestation case:** Can we detect a specific deforestation event (e.g. Amazon frontier, 2020-2022) using NDVI differencing at Sentinel-2 resolution? What threshold and date pair would be most effective?